# ML Customer Review Analysis

This notebook trains and demos a **customer-review sentiment classifier**. It
maps the free-text of a product review to one of **three sentiments** (positive,
neutral, negative) — and to a **binary** positive/negative label — using classic
NLP (TF-IDF) plus Logistic Regression, trained on the Flipkart product reviews
dataset.

It walks through the full lifecycle: load → baseline model → regularization →
model comparison (Naive Bayes, Decision Tree) → feature (n-gram) experiments →
final evaluation → threshold analysis → save → predict a review. Project-level
docs (setup, repo layout, usage) are in the `README.md`.

### Model used here

| Stage      | Component                                            |
| ---------- | ---------------------------------------------------- |
| Features   | `TfidfVectorizer(lowercase=True, ngram=(1,1))`       |
| Classifier | `LogisticRegression(C=10.0, max_iter=1000)`          |
| Task       | Binary text classification (positive / negative)     |

Evaluation uses **accuracy** and **macro-F1** to account for class imbalance.

---

## Notebook workflow

1. Load the dataset
2. Build a baseline logistic regression model
3. Add regularization (C)
4. Compare with Naive Bayes
5. Compare with Decision Tree
6. Feature engineering (n-grams)
7. Evaluate the final model
8. Threshold analysis
9. Train and save the final model
10. Predict a single review


## 1. Load the dataset

In [1]:
import pandas as pd
import numpy as np

url = "./flipkart_data.csv"

df = pd.read_csv(url)

# Step 1: Assign sentiment based on rating
conditions = [
    df["rating"] >= 4,
    df["rating"] == 3,
    df["rating"] <= 2
]

choices = ["positive", "neutral", "negative"]

df["sentiment"] = np.select(
    conditions,
    choices,
    default="neutral"
)

print("Original dataset size:", len(df))
print("\nOriginal sentiment distribution:")
print(df["sentiment"].value_counts())
print("\nOriginal sentiment proportions:")
print(df["sentiment"].value_counts(normalize=True).round(3))

# Step 2: Create a binary dataset (exclude neutral reviews)
df_binary = df[
    df["sentiment"].isin(["positive", "negative"])
].copy()

print("\nBinary dataset size:", len(df_binary))
print("\nBinary sentiment distribution:")
print(df_binary["sentiment"].value_counts())
print("\nBinary sentiment proportions:")
print(df_binary["sentiment"].value_counts(normalize=True).round(3))

# Step 3: Save both datasets
df.to_csv("./flipkart_data_with_sentiment.csv", index=False)
df_binary.to_csv("./flipkart_data_binary.csv", index=False)

print("\nSaved:")
print("- flipkart_data_with_sentiment.csv")
print("- flipkart_data_binary.csv")




Original dataset size: 9976

Original sentiment distribution:
sentiment
positive    8091
negative    1001
neutral      884
Name: count, dtype: int64

Original sentiment proportions:
sentiment
positive    0.811
negative    0.100
neutral     0.089
Name: proportion, dtype: float64

Binary dataset size: 9092

Binary sentiment distribution:
sentiment
positive    8091
negative    1001
Name: count, dtype: int64

Binary sentiment proportions:
sentiment
positive    0.89
negative    0.11
Name: proportion, dtype: float64

Saved:
- flipkart_data_with_sentiment.csv
- flipkart_data_binary.csv


## 2. Build a baseline logistic regression model

In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix

#load dataset

url = "./flipkart_data_binary.csv"
df = pd.read_csv(url)

#Split the dataset into training, evaluation and testing sets
X = df["review"]
y = df["sentiment"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
X_eval, X_test, y_eval, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# model
model = Pipeline([
    ('tfidf', TfidfVectorizer(lowercase=True, ngram_range=(1, 2), max_features=30000)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
])

model.fit(X_train, y_train)

predictions = model.predict(X_test)


# Evaluate the model
accuracy = accuracy_score(y_test, predictions)
f1 = f1_score(y_test, predictions, average='macro') # difference between macro and weighted is that macro calculates metrics for each label, and finds their unweighted mean. This does not take label imbalance into account. Weighted calculates metrics for each label, and finds their average weighted by support (the number of true instances for each label). This alters ‘macro’ to account for label imbalance; it can result in an F-score that is not between precision and recall.

print(f"Training examples: {len(X_train)}")
print(f"Validation examples: {len(X_test)}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Macro-F1: {f1:.4f}")

print(classification_report(
    y_test,
    predictions,
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_test, predictions))
print("Full dataset distribution:")
print(y.value_counts())
print(y.value_counts(normalize=True).round(3))


error_df = pd.DataFrame({
    "review": X_test,
    "actual": y_test,
    "predicted": predictions
})

# Negative reviews incorrectly predicted as positive
false_negatives = error_df[
    (error_df["actual"] == "negative") &
    (error_df["predicted"] == "positive")
]

print("Negative reviews predicted as positive:",
      len(false_negatives))

print(false_negatives["review"].sample(
    min(20, len(false_negatives)),
    random_state=42
).to_string(index=False))






Training examples: 5455
Validation examples: 1819
Accuracy: 0.9329
Macro-F1: 0.8453
              precision    recall  f1-score   support

    negative       0.66      0.82      0.73       200
    positive       0.98      0.95      0.96      1619

    accuracy                           0.93      1819
   macro avg       0.82      0.88      0.85      1819
weighted avg       0.94      0.93      0.94      1819

Confusion matrix:
[[ 164   36]
 [  86 1533]]
Full dataset distribution:
sentiment
positive    8091
negative    1001
Name: count, dtype: int64
sentiment
positive    0.89
negative    0.11
Name: proportion, dtype: float64
Negative reviews predicted as positive: 36
No more need to tell youBoat sound and bass alw...
It's nice products but suddenly it's stop after...
Voice quality and volume during call is very lo...
After using it for more than month here is a ho...
Sound quality is like as 60₹  headphones and al...
Calling Experience: 4/10Music Listening: 5/10No...
Product is very good 

## 3. Add regularization (C)

In [3]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix

#load dataset

url = "./flipkart_data_binary.csv"
df = pd.read_csv(url)

#Split the dataset into training, evaluation and testing sets
X = df["review"]
y = df["sentiment"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
X_eval, X_test, y_eval, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

for c in [0.1, 1.0, 10.0]:
    print(f"Training with regularization parameter C={c}")
    # model
    model = Pipeline([
        ('tfidf', TfidfVectorizer(lowercase=True, ngram_range=(1, 2), max_features=30000)),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced', C=c))
    ])

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    # Evaluate the model
    accuracy = accuracy_score(y_test, predictions)
    f1 = f1_score(y_test, predictions, average='macro') 

    print(f"Training examples: {len(X_train)}")
    print(f"Validation examples: {len(X_test)}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Macro-F1: {f1:.4f}")

    print(classification_report(
        y_test,
        predictions,
        zero_division=0
    ))



Training with regularization parameter C=0.1


Training examples: 5455
Validation examples: 1819
Accuracy: 0.9010
Macro-F1: 0.7981
              precision    recall  f1-score   support

    negative       0.53      0.85      0.65       200
    positive       0.98      0.91      0.94      1619

    accuracy                           0.90      1819
   macro avg       0.76      0.88      0.80      1819
weighted avg       0.93      0.90      0.91      1819

Training with regularization parameter C=1.0


Training examples: 5455
Validation examples: 1819
Accuracy: 0.9329
Macro-F1: 0.8453
              precision    recall  f1-score   support

    negative       0.66      0.82      0.73       200
    positive       0.98      0.95      0.96      1619

    accuracy                           0.93      1819
   macro avg       0.82      0.88      0.85      1819
weighted avg       0.94      0.93      0.94      1819

Training with regularization parameter C=10.0


Training examples: 5455
Validation examples: 1819
Accuracy: 0.9412
Macro-F1: 0.8551
              precision    recall  f1-score   support

    negative       0.71      0.78      0.74       200
    positive       0.97      0.96      0.97      1619

    accuracy                           0.94      1819
   macro avg       0.84      0.87      0.86      1819
weighted avg       0.94      0.94      0.94      1819



## 4. Compare with Naive Bayes

In [4]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

#load dataset

url = "./flipkart_data_binary.csv"
df = pd.read_csv(url)

#Split the dataset into training, evaluation and testing sets
X = df["review"]
y = df["sentiment"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
X_eval, X_test, y_eval, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)


for alpha in [0.1, 0.5, 1.0]:
    print(f"Training with smoothing parameter alpha={alpha}")
    # model
    model = Pipeline([
        ('tfidf', TfidfVectorizer(lowercase=True, ngram_range=(1, 2), max_features=30000)),
        ('classifier', MultinomialNB(alpha=alpha))
    ])
    # Calculate weights from training labels only
    sample_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_train
    )

    model.fit(X_train, y_train,classifier__sample_weight=sample_weights)

    predictions = model.predict(X_test)


    # Evaluate the model
    accuracy = accuracy_score(y_test, predictions)
    f1 = f1_score(y_test, predictions, average='macro') 
    
    print(f"Training examples: {len(X_train)}")
    print(f"Validation examples: {len(X_test)}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Macro-F1: {f1:.4f}")

    print(classification_report(
        y_test,
        predictions,
        zero_division=0
    ))

    print("Confusion matrix:")
    print(confusion_matrix(y_test, predictions))

Training with smoothing parameter alpha=0.1


Training examples: 5455
Validation examples: 1819
Accuracy: 0.9302
Macro-F1: 0.8342
              precision    recall  f1-score   support

    negative       0.66      0.77      0.71       200
    positive       0.97      0.95      0.96      1619

    accuracy                           0.93      1819
   macro avg       0.81      0.86      0.83      1819
weighted avg       0.94      0.93      0.93      1819

Confusion matrix:
[[ 154   46]
 [  81 1538]]
Training with smoothing parameter alpha=0.5


Training examples: 5455
Validation examples: 1819
Accuracy: 0.9137
Macro-F1: 0.8189
              precision    recall  f1-score   support

    negative       0.57      0.86      0.69       200
    positive       0.98      0.92      0.95      1619

    accuracy                           0.91      1819
   macro avg       0.78      0.89      0.82      1819
weighted avg       0.94      0.91      0.92      1819

Confusion matrix:
[[ 173   27]
 [ 130 1489]]
Training with smoothing parameter alpha=1.0


Training examples: 5455
Validation examples: 1819
Accuracy: 0.9038
Macro-F1: 0.8052
              precision    recall  f1-score   support

    negative       0.54      0.88      0.67       200
    positive       0.98      0.91      0.94      1619

    accuracy                           0.90      1819
   macro avg       0.76      0.89      0.81      1819
weighted avg       0.93      0.90      0.91      1819

Confusion matrix:
[[ 175   25]
 [ 150 1469]]


## 5. Compare with Decision Tree

In [5]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier

#load dataset

url = "./flipkart_data_binary.csv"
df = pd.read_csv(url)

#Split the dataset into training, evaluation and testing sets
X = df["review"]
y = df["sentiment"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
X_eval, X_test, y_eval, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# model
model = Pipeline([
    ('tfidf', TfidfVectorizer(lowercase=True, ngram_range=(1, 2), max_features=30000)),
    ("decision_tree", DecisionTreeClassifier( class_weight="balanced", random_state=42)),
])

model.fit(X_train, y_train)

predictions = model.predict(X_test)


# Evaluate the model
accuracy = accuracy_score(y_test, predictions)
f1 = f1_score(y_test, predictions, average='macro') 

print(f"Training examples: {len(X_train)}")
print(f"Validation examples: {len(X_test)}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Macro-F1: {f1:.4f}")

print(classification_report(
    y_test,
    predictions,
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_test, predictions))







Training examples: 5455
Validation examples: 1819
Accuracy: 0.9214
Macro-F1: 0.7996
              precision    recall  f1-score   support

    negative       0.64      0.65      0.64       200
    positive       0.96      0.96      0.96      1619

    accuracy                           0.92      1819
   macro avg       0.80      0.80      0.80      1819
weighted avg       0.92      0.92      0.92      1819

Confusion matrix:
[[ 129   71]
 [  72 1547]]


## 6. Feature engineering (n-grams)

In [6]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix

#load dataset

url = "./flipkart_data_binary.csv"
df = pd.read_csv(url)

#Split the dataset into training, evaluation and testing sets
X = df["review"]
y = df["sentiment"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
X_eval, X_test, y_eval, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

for ngram_range in [(1, 1), (1, 2), (1, 3)]:
    print(f"Training with ngram_range={ngram_range}")
    # model
    model = Pipeline([
        ('tfidf', TfidfVectorizer(lowercase=True, ngram_range=ngram_range, max_features=30000)),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced', C=10.0))
    ])

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    # Evaluate the model
    accuracy = accuracy_score(y_test, predictions)
    f1 = f1_score(y_test, predictions, average='macro') 

    print(f"Training examples: {len(X_train)}")
    print(f"Validation examples: {len(X_test)}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Macro-F1: {f1:.4f}")

    print(classification_report(
        y_test,
        predictions,
        zero_division=0
    ))

    print("Confusion matrix:")
    print(confusion_matrix(y_test, predictions))


Training with ngram_range=(1, 1)


Training examples: 5455
Validation examples: 1819
Accuracy: 0.9406
Macro-F1: 0.8609
              precision    recall  f1-score   support

    negative       0.69      0.83      0.76       200
    positive       0.98      0.95      0.97      1619

    accuracy                           0.94      1819
   macro avg       0.83      0.89      0.86      1819
weighted avg       0.95      0.94      0.94      1819

Confusion matrix:
[[ 167   33]
 [  75 1544]]
Training with ngram_range=(1, 2)


Training examples: 5455
Validation examples: 1819
Accuracy: 0.9412
Macro-F1: 0.8551
              precision    recall  f1-score   support

    negative       0.71      0.78      0.74       200
    positive       0.97      0.96      0.97      1619

    accuracy                           0.94      1819
   macro avg       0.84      0.87      0.86      1819
weighted avg       0.94      0.94      0.94      1819

Confusion matrix:
[[ 155   45]
 [  62 1557]]
Training with ngram_range=(1, 3)


Training examples: 5455
Validation examples: 1819
Accuracy: 0.9423
Macro-F1: 0.8572
              precision    recall  f1-score   support

    negative       0.72      0.78      0.75       200
    positive       0.97      0.96      0.97      1619

    accuracy                           0.94      1819
   macro avg       0.85      0.87      0.86      1819
weighted avg       0.94      0.94      0.94      1819

Confusion matrix:
[[ 155   45]
 [  60 1559]]


## 7. Evaluate the final model

In [7]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix

#load dataset

url = "./flipkart_data_binary.csv"
df = pd.read_csv(url)

#Split the dataset into training, evaluation and testing sets
X = df["review"]
y = df["sentiment"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
X_eval, X_test, y_eval, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# model
model = Pipeline([
    ('tfidf', TfidfVectorizer(lowercase=True, ngram_range=(1, 1), max_features=30000)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced', C=10.0))
])

model.fit(X_train, y_train)
predictions = model.predict(X_eval)

# Evaluate the model
accuracy = accuracy_score(y_eval, predictions)
f1 = f1_score(y_eval, predictions, average='macro') 

print(f"Training examples: {len(X_train)}")
print(f"Validation examples: {len(X_eval)}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Macro-F1: {f1:.4f}")

print(classification_report(
    y_eval,
    predictions,
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_eval, predictions))


Training examples: 5455
Validation examples: 1818
Accuracy: 0.9400
Macro-F1: 0.8599
              precision    recall  f1-score   support

    negative       0.69      0.83      0.75       200
    positive       0.98      0.95      0.97      1618

    accuracy                           0.94      1818
   macro avg       0.83      0.89      0.86      1818
weighted avg       0.95      0.94      0.94      1818

Confusion matrix:
[[ 167   33]
 [  76 1542]]


## 8. Threshold analysis

In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix, precision_score, recall_score

#load dataset

url = "./flipkart_data_binary.csv"
df = pd.read_csv(url)

#Split the dataset into training, evaluation and testing sets
X = df["review"]
y = df["sentiment"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
X_eval, X_test, y_eval, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# model
model = Pipeline([
    ('tfidf', TfidfVectorizer(lowercase=True, ngram_range=(1, 1), max_features=30000)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced', C=10.0))
])

model.fit(X_train, y_train)
predictions = model.predict(X_eval)

# Evaluate the model
accuracy = accuracy_score(y_eval, predictions)
f1 = f1_score(y_eval, predictions, average='macro') 

print(f"Training examples: {len(X_train)}")
print(f"Validation examples: {len(X_eval)}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Macro-F1: {f1:.4f}")

print(classification_report(
    y_eval,
    predictions,
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_eval, predictions))


negative_index = list(model.classes_).index("negative")
negative_probabilities = model.predict_proba(X_eval)[:, negative_index]

# Apply the selected threshold
test_predictions = np.where(
    negative_probabilities >= 0.70,
    "negative",
    "positive"
)

# Evaluate on the test labels
print(f"Test examples: {len(X_eval)}")
print(f"Test accuracy: {accuracy_score(y_eval, test_predictions):.4f}")
print(f"Test macro-F1: {f1_score(y_eval, test_predictions, average='macro'):.4f}")

print("\nTest classification report:")
print(classification_report(y_eval, test_predictions, zero_division=0))

print("Test confusion matrix:")
print(confusion_matrix(y_eval, test_predictions))


Training examples: 5455
Validation examples: 1818
Accuracy: 0.9400
Macro-F1: 0.8599
              precision    recall  f1-score   support

    negative       0.69      0.83      0.75       200
    positive       0.98      0.95      0.97      1618

    accuracy                           0.94      1818
   macro avg       0.83      0.89      0.86      1818
weighted avg       0.95      0.94      0.94      1818

Confusion matrix:
[[ 167   33]
 [  76 1542]]
Test examples: 1818
Test accuracy: 0.9505
Test macro-F1: 0.8713

Test classification report:
              precision    recall  f1-score   support

    negative       0.79      0.76      0.77       200
    positive       0.97      0.97      0.97      1618

    accuracy                           0.95      1818
   macro avg       0.88      0.86      0.87      1818
weighted avg       0.95      0.95      0.95      1818

Test confusion matrix:
[[ 151   49]
 [  41 1577]]


## 9. Train and save the final model

In [9]:
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

url = "./flipkart_data_binary.csv"
df = pd.read_csv(url)

#Split the dataset into training, evaluation and testing sets
X = df["review"]
y = df["sentiment"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
X_eval, X_test, y_eval, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# model
model = Pipeline([
    ('tfidf', TfidfVectorizer(lowercase=True, ngram_range=(1, 1), max_features=30000)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced', C=10.0))
])

model.fit(X_train, y_train)
print("Model training completed.")


# 4. Save the trained model to disk
model_path = "customer_review_model.joblib"
joblib.dump(model, model_path)
print(f"Trained model saved to: {model_path}")

Model training completed.
Trained model saved to: customer_review_model.joblib


## 10. Predict a single review

In [10]:
import joblib

model = joblib.load("customer_review_model.joblib")


def predict_review(text):
    if not text or not text.strip():
        raise ValueError("Review text cannot be empty.")

    predicted_intent = model.predict([text])[0]
    probabilities = model.predict_proba([text])[0]
    confidence = float(max(probabilities))

    return {
        "review_text": text,
        "predicted_intent": predicted_intent,
        "confidence": confidence,
    }


# Change this review to predict any other review.
review = "The product quality is amazing and worth every rupee"

result = predict_review(review)

print("\nPrediction")
print(f"Review: {result['review_text']}")
print(f"Intent: {result['predicted_intent']}")
print(f"Confidence: {result['confidence']:.2%}")



Prediction
Review: The product quality is amazing and worth every rupee
Intent: positive
Confidence: 77.09%
